# Leksjon 10 - AI-agenter i produksjon

I denne leksjonen vil du lære **produksjonsmønstre** for AI-agenter ved bruk av Microsoft Agent Framework (Python). Vi dekker:

- **Observerbarhet** — legge til tidsmåling og logging for agentinteraksjoner
- **Evaluering** — bruke en evalueringsagent for å vurdere svarenes kvalitet
- **Kostnadsstyring** — strategier for tokenoptimalisering og modellvalg

Scenariet er en **reiseagent** som hjelper brukere med å planlegge reiser, med overvåking og evaluering lagt på toppen.


## Oppsett


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Produksjonsbetraktninger

Å flytte AI-agenter fra prototyper til produksjon krever nøye oppmerksomhet til tre søyler:

1. **Observabilitet** — Du trenger innsyn i hva agenten gjør, hvor lang tid det tar, og hvilke verktøy den kaller. Uten sporing og logging er det nesten umulig å feilsøke problemer i produksjon.

2. **Evaluering** — Automatiserte kvalitetskontroller sørger for at agentens svar forblir nøyaktige, fullstendige og hjelpsomme over tid. En evalueringsagent kan gi poeng til svarene ut fra definerte kriterier.

3. **Kostnadsstyring** — Tokenbruk påvirker kostnadene direkte. Strategier som promptoptimalisering, modellvalg og caching hjelper med å holde utgiftene under kontroll uten å gå på bekostning av kvaliteten.


## Bygge en observerbar agent

Vi definerer reiseverktøy og pakker agentkallet inn med tidsmåling slik at vi kan overvåke forsinkelse. I produksjon ville du integrert med OpenTelemetry eller en lignende sporings-backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evalueringsmønstre

Et vanlig produksjonsmønster er å bruke en sekundær agent som en **evaluatør**. Evaluatøren gir poeng til hovedagentens svar basert på forhåndsdefinerte kriterier som fullstendighet, nøyaktighet og hjelpsomhet.

Dette muliggjør:
- Automatiserte kvalitetskontroller før svar når brukerne
- Oppdagelse av regresjoner når forespørsler eller modeller endres
- Kontinuerlig overvåking av agentens ytelse over tid


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategier for kostnadsstyring

Å kontrollere kostnader er avgjørende for produksjons-AI-agenter. Her er viktige strategier:

| Strategi | Beskrivelse |
|---|---|
| **Prompt-optimalisering** | Hold systeminstruksjoner korte. Fjern overflødig kontekst for å redusere antall input-tokens. |
| **Modellvalg** | Bruk mindre, billigere modeller (f.eks. GPT-4o-mini) for enkle oppgaver som klassifisering eller ekstraksjon, og reserver større modeller for komplekst resonnement. |
| **Bufring** | Bufre verktøyresultater og hyppige spørringer for å unngå unødvendige API-kall. |
| **Tokenbudsjetter** | Angi `max_tokens`-grenser for å forhindre uventet lange svar. |
| **Batching** | Gruppér flere brukerforespørsler i ett enkelt API-kall der det er mulig. |

I praksis fungerer en lagdelt tilnærming godt: rute enkle forespørsler til en rask, rimelig modell og eskaler kun komplekse forespørsler til en mer kapabel modell.


## Sammendrag

I denne leksjonen lærte du hvordan du:

1. **Legge til observabilitet** i agentinteraksjoner med tidsmåling og logging, og legge grunnlaget for sporing og overvåking.
2. **Vurdere agentresponsene** automatisk ved å bruke en evalueringsagent som gir poeng for fullstendighet, nøyaktighet og hjelpsomhet.
3. **Håndtere kostnader** gjennom promptoptimalisering, modellvalg, caching og tokenbudsjetter.

Disse produksjonsmønstrene hjelper med å sikre at AI-agentene dine er pålitelige, målbare, og kostnadseffektive i stor skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfraskrivelse:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Originaldokumentet på det opprinnelige språket bør betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
